# Single layer testing

This notebook tests the backend setup for a single layer.

In [ ]:
import cases
import eradiate
import matplotlib.pyplot as plt
import seaborn as sns
from util import Result, reshape_pplane

import eradiate_disort as ed

sns.set_theme(style="ticks")

# eradiate.set_mode("mono")
eradiate.set_mode("ckd")

SPP = 10_000
if eradiate.get_mode().is_ckd:
    SPP //= 16
PHASES = ["isotropic", "rayleigh"]

In [ ]:
results = {}

for phase in PHASES:
    exp = cases.single_layer(30.0, phase)
    result = Result()

    backend = ed.EradiateDisortBackend()
    result.disort = reshape_pplane(backend.run(exp))
    result.mitsuba = eradiate.run(exp, spp=SPP)["radiance"].squeeze()

    results[phase] = result

In [ ]:
def plot(results):
    fig, axs = plt.subplots(1, len(results), figsize=(8, 3.5), layout="constrained")

    for k, phase in enumerate(results.keys()):
        ax = axs[k]
        result = results[phase]

        ax.plot(
            result.mitsuba["vza"],
            result.mitsuba,
            label="Mitsuba" if k == 0 else None,
        )
        ax.plot(
            result.disort["vza"],
            result.disort,
            label="CDISORT" if k == 0 else None,
            ls="--",
        )

        ax.set_xlabel("θ [°]")
        ax.set_ylabel("Radiance [W/m²/sr]" if k == 0 else None)
        ax.set_title(phase.title())

    fig.legend(title="Backend", loc="outside upper center", ncol=2)

    return fig, axs


plot(results)
plt.show()
plt.close()